# Testing the Integrated Simulation Platform
This notebook contains codes to test the various functionalities of the integrated simulation platform. You can call any of the modules from the package here. 

In [ ]:
%load_ext autoreload
%autoreload 2

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## Import required packages

In [2]:
from pathlib import Path
from infrarisk.src.network_recovery import *
import infrarisk.src.simulation as simulation
from infrarisk.src.physical.integrated_network import *
import infrarisk.src.recovery_strategies as strategies
import infrarisk.src.socioeconomic.se_analysis as se_analysis
from infrarisk.src.physical.interdependencies import *

import infrarisk.src.plots as model_plots
#hide warnings
import warnings
warnings.filterwarnings('ignore')

## Create an IntegratedNetwork object

In [3]:
shelby_network = IntegratedNetwork(name = "Shelby")

### Load the three infrastructure models: Water, Power and Transportation

Three different models are used:
- Water distribution network using **wntr** package
- Power systems using **pandapower** package
- Transportation network using static traffic assignment package developed by Dr. Stephen Boyles (University of Texas at Austin)

In [ ]:
MAIN_DIR = Path('../..')
SIM_STEP = 60

network_dir= 'infrarisk/data/networks/shelby'
water_folder = MAIN_DIR/f'{network_dir}/water'
power_folder = MAIN_DIR/f'{network_dir}/power'
transp_folder = MAIN_DIR/f'{network_dir}/transportation/reduced'

# load all infrastructure networks
shelby_network.load_networks(water_folder=water_folder, 
                             power_folder=power_folder, 
                             transp_folder=transp_folder,
                             sim_step=SIM_STEP)

### Create a Networkx graph of the integrated infrastructure network.

In [ ]:
shelby_network.generate_integrated_graph(basemap = True)

### Build interdependencies

Three types of dependencies:
- Power - Water dependencies (eg.: water pump on electric motor, generator on reservoir)
- Power - Transportation dependencies (eg.: road access to power system components for M&R)
- Water - Transportation dependencies (eg.: road access to water network components for M&R)

The dependencies are referenced using two tables in the model.
- **wp_table** for water - power dependencies
- **access_table** for transportation dependencies

In [ ]:
dependency_file = MAIN_DIR/f"{network_dir}/dependencies.csv"
shelby_network.generate_dependency_table(dependency_file = dependency_file)
shelby_network.dependency_table.wp_table.head()

In [ ]:
shelby_network.dependency_table.access_table.head()

# Socioeconomic data for Shelby County

In [8]:
year, tract, county, state = 2017, '*', 157, 47
county = 157
se_dir = MAIN_DIR/f"{network_dir}/gis/se_data"
if not os.path.exists(se_dir):
    os.makedirs(se_dir)

ShelbySE = se_analysis.SocioEconomicTable(name = 'Shelby', year = year, 
                                               tract = tract, state = state, 
                                               county = county, dir = se_dir)

ShelbySE.download_se_data(force_download = False)
ShelbySE.create_setable()

In [ ]:
ShelbySE.plot_interactive(type = "annual receipts")

# Define disaster scenario

### Set failed components

In [ ]:
scenario_folder = f"scenarios/scenario1"
disruption_file = MAIN_DIR/f"{network_dir}/{scenario_folder}/disruption_file.dat"

shelby_network.set_disrupted_components(disruption_file=disruption_file)
disrupted_components = shelby_network.get_disrupted_components()
print(*disrupted_components, sep = ", ")

### Set initial crew locations

In [ ]:
crew_count = 10
shelby_network.deploy_crews(
    init_power_crew_locs=['T_J8']*crew_count, 
    init_water_crew_locs=['T_J8']*crew_count,
    init_transpo_crew_locs= ['T_J8']*crew_count
    )

## Simulation of interdependent effects using a test scenario
### (a) Create NetworkRecovery

In [12]:
network_recovery = NetworkRecovery(shelby_network, 
                                   sim_step=SIM_STEP, 
                                   pipe_close_policy="repair",
                                   pipe_closure_delay= 12*60, 
                                   line_close_policy="sensor_based_line_isolation",
                                   line_closure_delay= 12*60)

### (b) Create a simulation object

In [13]:
bf_simulation = simulation.NetworkSimulation(network_recovery)

### (c) Generation of random repair order

In [14]:
capacity_strategy = strategies.HandlingCapacityStrategy(shelby_network)
capacity_strategy.set_repair_order()
repair_order = capacity_strategy.get_repair_order()

#repair_order = ['P_L46', 'P_L45', 'W_PMA53', 'W_PMA44', 'P_L54', 'P_L38', 'P_L43', 'P_L2']

import os
if not os.path.exists(MAIN_DIR/f"{network_dir}/{scenario_folder}/capacity"):
    os.makedirs(MAIN_DIR/f"{network_dir}/{scenario_folder}/capacity")

In [ ]:
#Generate a random repair order
# repair_order = network_recovery.network.get_disrupted_components()
# random.shuffle(repair_order)
print('Current repair order is {}'.format(repair_order))

### (d) Generation of event tables

In [ ]:
bf_simulation.network_recovery.schedule_recovery(repair_order)

In [17]:
#bf_simulation.network_recovery.event_table.to_csv("event_tbl.csv", sep = "\t", index = False)
bf_simulation.expand_event_table()

### (e) Simulation of interdependent effects

In [ ]:
resilience_metrics = bf_simulation.simulate_interdependent_effects(
    bf_simulation.network_recovery)

In [ ]:
strategy = 'capacity'
bf_simulation.write_results(f"{MAIN_DIR}/{network_dir}/{scenario_folder}/{strategy}", 
                            resilience_metrics)

### (f) Calculation of resilience metric

In [20]:
resilience_metrics.calculate_power_resmetric(network_recovery)

In [21]:
resilience_metrics.calculate_water_resmetrics(network_recovery)

In [ ]:
resilience_metrics.set_weighted_auc_metrics()

In [ ]:
resilience_metrics.weighed_pcs_auc

In [24]:
ShelbySE.combine_infrastructure_se_data(shelby_network, resilience_metrics)
ShelbySE.calculate_economic_costs()

In [ ]:
ShelbySE.economic_cost_df['00'].sum()

In [26]:
ShelbySE.economic_cost_df.to_csv(f"{MAIN_DIR}/{network_dir}/{scenario_folder}/{strategy}/economic_cost.csv", index=False)

# Plot network performance during the disruption
### Overall system performance considering indirect effects

In [27]:
model_plots.plot_interdependent_effects(resilience_metrics, metric = 'pcs', title = False)

### Location of disrupted components and crews

In [ ]:
model_plots.plot_disruptions_and_crews(shelby_network, basemap = True)

### Disruption to utility services

In [ ]:
split_water_sa = gpd.overlay(shelby_network.wn.service_area, ShelbySE.county_gpd_truncated, how='intersection')
split_power_sa = gpd.overlay(shelby_network.pn.service_area, ShelbySE.county_gpd_truncated, how='intersection')
sa_dict = {'Water': split_water_sa, 'Power': split_power_sa}

model_plots.plot_region_impact_map(resilience_metrics, sa_dict, "capacity", extends = ShelbySE.bounds)

### Direct business disruptions

In [ ]:
ShelbySE.plot_interactive(type = "economic costs")